# FD001 Final Validation Selection

Objetivo: juntar resultados de validación artificial y elegir un único candidato sin mirar el test oficial.


In [ ]:
from collections import OrderedDict
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
PREDICTIONS_DIR = PROJECT_ROOT / "predictions"
for path in [RESULTS_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
DATA_DIR = PROJECT_ROOT / "CMAPSSData"
EVAL_SIZE = 0.2
CUT_RULS = (20, 50, 80, 110, 140)
DEFAULT_WINDOW_SIZE = 30
DEFAULT_RUL_CAP = 125

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

VALIDATION_FILES = [
    ("temporal_baseline", RESULTS_DIR / "fd001_temporal_metrics.csv", RESULTS_DIR / "fd001_temporal_metrics_by_rul_bin.csv"),
    ("boosting", RESULTS_DIR / "fd001_boosting_metrics.csv", RESULTS_DIR / "fd001_boosting_metrics_by_rul_bin.csv"),
    ("window_cap_search", RESULTS_DIR / "fd001_window_cap_search.csv", None),
    ("mlp", RESULTS_DIR / "fd001_mlp_metrics.csv", RESULTS_DIR / "fd001_mlp_metrics_by_rul_bin.csv"),
    ("sample_weight", RESULTS_DIR / "fd001_sample_weight_metrics.csv", RESULTS_DIR / "fd001_sample_weight_metrics_by_rul_bin.csv"),
]


def bin_wide(bin_path):
    if bin_path is None or not bin_path.exists():
        return None
    bins = pd.read_csv(bin_path)
    rows = []
    for keys, group in bins.groupby(["representation", "model_name"], sort=False):
        representation, model_name = keys
        row = {"representation": representation, "model_name": model_name}
        for _, item in group.iterrows():
            label = str(item["rul_bin"]).replace("-", "_").replace("+", "plus")
            row[f"mae_rul_{label}"] = item["mae"]
            row[f"dangerous_error_pct_rul_{label}"] = item["dangerous_error_pct"]
        rows.append(row)
    return pd.DataFrame(rows)


def load_validation_table(source, metrics_path, bins_path=None):
    if not metrics_path.exists():
        print(f"Falta {metrics_path.name}; se omite {source}.")
        return None
    table = pd.read_csv(metrics_path)
    table.insert(0, "source", source)
    wide = bin_wide(bins_path) if bins_path is not None else None
    if wide is not None:
        missing_cols = [c for c in wide.columns if c not in table.columns]
        table = table.merge(wide[["representation", "model_name", *missing_cols]], on=["representation", "model_name"], how="left")
    return table


In [ ]:
tables = [load_validation_table(*item) for item in VALIDATION_FILES]
tables = [table for table in tables if table is not None]
all_metrics = pd.concat(tables, ignore_index=True, sort=False)

required = ["rmse", "mae", "r2", "cmapss_score_mean", "dangerous_error_pct"]
for col in required:
    all_metrics[col] = pd.to_numeric(all_metrics[col], errors="coerce")

for col in ["mae_rul_0_30", "mae_rul_30_60", "mae_rul_60_90", "mae_rul_90plus"]:
    if col not in all_metrics.columns:
        all_metrics[col] = np.nan
    all_metrics[col] = pd.to_numeric(all_metrics[col], errors="coerce")

ranked = all_metrics.dropna(subset=["rmse", "mae", "cmapss_score_mean", "dangerous_error_pct"]).copy()
ranked["rank_rmse"] = ranked["rmse"].rank(method="min")
ranked["rank_mae"] = ranked["mae"].rank(method="min")
ranked["rank_cmapss"] = ranked["cmapss_score_mean"].rank(method="min")
ranked["rank_dangerous"] = ranked["dangerous_error_pct"].rank(method="min")
ranked["rank_near_failure"] = ranked["mae_rul_0_30"].fillna(ranked["mae"]).rank(method="min")
ranked["rank_mid_rul"] = ranked["mae_rul_60_90"].fillna(ranked["mae"]).rank(method="min")
ranked["validation_selection_score"] = (
    ranked["rank_rmse"]
    + ranked["rank_mae"]
    + ranked["rank_cmapss"]
    + 1.5 * ranked["rank_dangerous"]
    + 1.5 * ranked["rank_near_failure"]
    + 0.5 * ranked["rank_mid_rul"]
)
ranked = ranked.sort_values(["validation_selection_score", "rmse", "dangerous_error_pct"]).reset_index(drop=True)
ranked.to_csv(RESULTS_DIR / "fd001_final_validation_ranking.csv", index=False)
ranked.head(20)


## Conclusión de validación


In [ ]:
figure_dir = FIGURES_DIR / "fd001_final_validation"
figure_dir.mkdir(parents=True, exist_ok=True)

top = ranked.head(15).copy()
top["label"] = top["source"] + " | " + top["model_name"].astype(str) + " | " + top["representation"].astype(str)
plt.figure(figsize=(10, 6))
sns.barplot(data=top, y="label", x="validation_selection_score", color="steelblue")
plt.xlabel("Ranking balanceado de validación")
plt.ylabel("Candidato")
plt.tight_layout()
plt.savefig(figure_dir / "final_validation_ranking.png", dpi=150)
plt.close()

best_rmse = ranked.sort_values(["rmse", "mae"]).iloc[0]
best_danger = ranked.sort_values(["dangerous_error_pct", "rmse"]).iloc[0]
best_near = ranked.sort_values(["mae_rul_0_30", "rmse"]).iloc[0]
recommended = ranked.iloc[0]

print("Mejor por RMSE:")
display(best_rmse[["source", "representation", "model_name", "rmse", "mae", "dangerous_error_pct", "mae_rul_0_30"]])
print("Mejor por errores peligrosos:")
display(best_danger[["source", "representation", "model_name", "rmse", "mae", "dangerous_error_pct", "mae_rul_0_30"]])
print("Mejor cerca de falla RUL <= 30:")
display(best_near[["source", "representation", "model_name", "rmse", "mae", "dangerous_error_pct", "mae_rul_0_30"]])
print("Candidato recomendado final por ranking balanceado:")
display(recommended[["source", "representation", "model_name", "rmse", "mae", "cmapss_score_mean", "dangerous_error_pct", "mae_rul_0_30", "validation_selection_score"]])

print(RESULTS_DIR / "fd001_final_validation_ranking.csv")
print(figure_dir)
